In [10]:
import pandas as pd
from tqdm import tqdm

from alphalab.data import HDFData
from alphalab.data import FactorDuckDB

In [3]:
TABLE2FACTOR = {
    "ElementaryFactor": [
        "复权开盘价", "复权最高价", "复权最低价", "复权收盘价", "复权均价",
        "开盘价", "最高价", "最低价", "收盘价", "均价", "复权因子",
        "成交量", "成交金额", "换手率",
        "涨跌停", "涨跌停_一字板",
        "流通市值", "总市值",
        "上市天数", "是否在市", "交易状态",
        "中信行业", "申万行业", "特殊处理", 
    ],
    # "IndexConstituentFactor": ["上证50成份权重", "沪深300成份权重", "中证500成份权重", "中证800成份权重", "中证1000成份权重", "中证2000成份权重"],
    # "BarraFactor": ["Beta", "BookToPrice"],
}

In [4]:
h5_dir = "D:/CPResearch/market_data/XYQuantData/HDF5Data"


h5 = HDFData(f"{h5_dir}/ElementaryFactor/是否在市.hdf5")
df_islisted = h5.fetch_df().stack()
df_universe = df_islisted[df_islisted == 1]

for table_name, factor_columns in TABLE2FACTOR.items():
    pbar = tqdm(factor_columns, desc=f"正在处理 {table_name} 表")
    df_fct_list = []
    for factor_name in pbar:
        pbar.set_description(f"正在处理 {table_name}.{factor_name}")
        h5 = HDFData(f"{h5_dir}/{table_name}/{factor_name}.hdf5")
        df_fct = h5.fetch_df().stack().reindex(df_universe.index)
        df_fct.name = factor_name
        df_fct_list.append(df_fct)

正在处理 ElementaryFactor.特殊处理: 100%|██████████| 24/24 [1:51:00<00:00, 277.52s/it]  


In [8]:
df_table = pd.concat(df_fct_list, axis=1).reset_index()
df_table = df_table.rename(columns={"level_0":"date", "level_1":"code"})
df_table

,date,code,复权开盘价,复权最高价,复权最低价,复权收盘价,复权均价,开盘价,最高价,最低价,...,涨跌停,涨跌停_一字板,流通市值,总市值,上市天数,是否在市,交易状态,中信行业,申万行业,特殊处理
0,2005-01-04,000001.SZ,164.851236,164.851236,161.599239,163.100160,162.887530,6.59,6.59,6.46,...,0.0,NaN,9.189040e+05,1.268676e+06,5026.0,1.0,交易,银行,金融服务,NaN
1,2005-01-04,000002.SZ,149.422312,151.998559,147.991064,150.853560,150.713298,5.22,5.31,5.17,...,0.0,NaN,8.309987e+05,1.198202e+06,5090.0,1.0,交易,房地产,房地产,NaN
2,2005-01-04,000004.SZ,21.069564,21.194605,20.381833,20.944522,20.988286,6.74,6.78,6.52,...,0.0,NaN,2.791034e+04,5.626438e+04,5105.0,1.0,交易,医药,医药生物,NaN
3,2005-01-04,000005.SZ,12.356804,12.356804,11.795131,11.963633,12.066419,2.20,2.20,2.10,...,0.0,NaN,7.965127e+04,1.509449e+05,5140.0,1.0,交易,房地产,房地产,ST
4,2005-01-04,000006.SZ,17.733374,18.163796,17.604248,18.077711,17.902099,4.12,4.22,4.09,...,0.0,NaN,6.627896e+04,1.065085e+05,4636.0,1.0,交易,房地产,房地产,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15975930,2026-04-24,920978.BJ,29.794744,29.591184,29.591184,29.591184,29.747925,27.81,28.30,27.45,...,0.0,NaN,2.787823e+05,4.972386e+05,940.0,1.0,交易,汽车,汽车,NaN
15975931,2026-04-24,920981.BJ,57.370720,54.903779,54.903779,54.903779,57.792825,40.00,42.48,38.24,...,0.0,NaN,1.428212e+05,2.927195e+05,1622.0,1.0,交易,电子,电子,NaN
15975932,2026-04-24,920982.BJ,336.349082,334.777842,334.777842,334.777842,341.218374,194.80,204.30,191.25,...,0.0,NaN,1.064555e+06,2.231002e+06,1010.0,1.0,交易,医药,美容护理,NaN
15975933,2026-04-24,920985.BJ,7.316050,7.130142,7.130142,7.130142,7.185367,6.69,6.72,6.51,...,0.0,NaN,1.429952e+05,2.017785e+05,1356.0,1.0,交易,电力设备及新能源,电力设备,NaN


In [13]:
db_path = "data/stock.duckdb"
with FactorDuckDB(db_path) as db:
    db.write_table_frame(table_name, df_table, replace=True)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))